In [1]:
import pathlib
import sys

_root = pathlib.Path().resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
if str(_root.parent) not in sys.path and (_root.parent / 'src').exists():
    sys.path.insert(0, str(_root.parent))


# 03 Product & Category Intelligence

**Research question:** How do purchasing patterns and customer review scores differ across product categories?
**Sources:** `analytics.fact_order_items` (item grain), `analytics.fact_reviews` (review-order grain). **Filter:** all statuses. **Review metric:** average order-level review among orders containing the category, computed at distinct `(order_id, category)` grain to avoid item-weighting within category and to document multi-category orders.

In [2]:
from src.db import get_connection

conn = get_connection(read_only=True)


In [3]:
cat = conn.execute("""
WITH valid_review AS (
  SELECT review_id, order_id, valid_review_score
  FROM analytics.fact_reviews
  WHERE valid_review_score IS NOT NULL
),
order_category AS (
  SELECT DISTINCT oi.order_id, COALESCE(oi.product_category_name, 'unknown') AS category
  FROM analytics.fact_order_items oi
),
item_metrics AS (
  SELECT COALESCE(oi.product_category_name, 'unknown') AS category,
         COUNT(*) AS items,
         COUNT(DISTINCT oi.order_id) AS orders,
         SUM(oi.price) AS item_revenue,
         AVG(oi.price) AS avg_item_price
  FROM analytics.fact_order_items oi
  GROUP BY 1
),
review_metrics AS (
  SELECT oc.category,
         COUNT(DISTINCT vr.order_id) AS orders_with_review,
         AVG(vr.valid_review_score) AS avg_review
  FROM order_category oc
  JOIN valid_review vr ON oc.order_id = vr.order_id
  GROUP BY oc.category
)
SELECT i.category, i.items, i.orders, i.item_revenue, i.avg_item_price,
       r.orders_with_review, r.avg_review
FROM item_metrics i
LEFT JOIN review_metrics r ON i.category = r.category
ORDER BY i.item_revenue DESC
""").fetchdf()
cat.head(20)

,category,items,orders,item_revenue,avg_item_price,orders_with_review,avg_review
0,beleza_saude,9670,8836,1258681.34,130.163531,8771,4.182189
1,relogios_presentes,5991,5624,1205005.68,201.135984,5576,4.065700
2,cama_mesa_banho,11115,9417,1036988.68,93.296327,9313,3.968617
3,esporte_lazer,8641,7720,988048.97,114.344285,7669,4.168242
4,informatica_acessorios,7827,6689,911954.32,116.513903,6649,4.025813
5,moveis_decoracao,8334,6449,729762.49,87.564494,6398,4.006040
6,cool_stuff,3796,3632,635290.85,167.357969,3599,4.169299
7,utilidades_domesticas,6964,5884,632248.66,90.788148,5843,4.142370
8,automotivo,4235,3897,592720.11,139.957523,3877,4.090629
9,ferramentas_jardim,4347,3518,485256.46,111.630196,3496,4.134341


Interpretation: review averages reflect orders containing the category, not item-level satisfaction and not a causal product-quality estimate.

In [4]:
conn.close()
